# vidaudit — cross-model hallucination-audit benchmark

This notebook is the project's **reproduction artifact**. It runs a small **cross-model benchmark**: which VLM is the better hallucination auditor, and does auditing a model's output *with the same model* create a blind spot?

1. Streams videos from [FineVideo](https://huggingface.co/datasets/HuggingFaceFV/finevideo) and extracts ground-truth chapter descriptions.
2. Builds a labeled dataset: **plausible synthetic mutations** (object swap, attribute change, entity injection) + a **real-hallucination subset** harvested by captioning frames with Qwen.
3. Audits every sample with **two verifiers** — open-weight **Qwen2.5-VL-3B** and **Gemini 2.5 Flash** — plus the text-comparison **baseline**.
4. Reports precision / recall / F1 **split by subset and verifier**, with a confidence sweep per verifier.

**The two findings to look for:**
- **Synthetic subset** (description = mutated narration, model-independent): a clean *"which VLM audits best"* comparison.
- **Real subset** (captions generated by Qwen): the Qwen verifier is a **self-audit** and Gemini is a **cross-audit** — the gap measures the *self-consistency blind spot* (a model tends to rubber-stamp its own hallucinations).

**Runtime:** T4 GPU. **Reproducibility:** FineVideo is gated (HF login); set `GEMINI_API_KEY` for the cross-check; pin the Qwen revision (a commit SHA) when reporting numbers.

## 1. Install + authenticate

In [ ]:
!git clone https://github.com/shaneopatrick/vidaudit.git /content/vidaudit
%cd /content/vidaudit
!pip install -q -e '.[qwen,eval]'
!python -m spacy download en_core_web_sm

In [ ]:
# FineVideo is a gated dataset — accept terms at
# https://huggingface.co/datasets/HuggingFaceFV/finevideo then log in.
from huggingface_hub import notebook_login

notebook_login()

## 2. Stream FineVideo → chapters + on-disk videos

We iterate the stream manually so each video's `.mp4` is written to disk with a `video_id` that matches the chapters extracted from its metadata — the frame sampler needs the file on disk.

In [ ]:
from pathlib import Path

from datasets import load_dataset

from eval.finevideo_loader import FineVideoChapter

LIMIT = 5
VIDEOS = Path('/content/videos')
VIDEOS.mkdir(exist_ok=True)


def _tc(v):
    """Parse seconds from a number or an 'HH:MM:SS.mmm' timecode."""
    if isinstance(v, (int, float)):
        return float(v)
    if not isinstance(v, str) or not v.strip():
        return None
    s = v.strip()
    try:
        return float(s)
    except ValueError:
        pass
    sec = 0.0
    for p in s.split(':'):
        try:
            sec = sec * 60 + float(p)
        except ValueError:
            return None
    return sec


def _scene_desc(scene):
    if scene.get('description'):
        return scene['description']
    acts = scene.get('activities') or []
    joined = ' '.join(a.get('description', '') for a in acts if isinstance(a, dict)).strip()
    return joined or scene.get('title', '')


def chapters_from_row(row, video_id):
    meta = row.get('json') or {}
    content = meta.get('content_metadata', meta)
    out = []
    for sc in content.get('scenes', []):
        ts = sc.get('timestamps') if isinstance(sc.get('timestamps'), dict) else {}
        start = _tc(ts.get('start_timestamp'))
        desc = _scene_desc(sc)
        if start is None or not desc:
            continue
        out.append(FineVideoChapter(
            video_id=video_id, timestamp_start=start,
            timestamp_end=_tc(ts.get('end_timestamp')), description=desc))
    return out


def _video_bytes(row):
    v = row.get('mp4') or row.get('video')
    if isinstance(v, dict):
        return v.get('bytes')
    return v if isinstance(v, (bytes, bytearray)) else None


ds = load_dataset('HuggingFaceFV/finevideo', split='train', streaming=True)
chapters = []
for i, row in enumerate(ds):
    if i >= LIMIT:
        break
    vid = f'video_{i}'
    data = _video_bytes(row)
    if data:
        (VIDEOS / f'{vid}.mp4').write_bytes(data)
    chapters.extend(chapters_from_row(row, vid))
print(f'{len(chapters)} chapters')
if chapters:
    print(chapters[0].model_dump())

If you got **0 chapters**, FineVideo's metadata layout may differ from what the cell above expects. Inspect one row's `json` and adjust the key paths:

```python
import itertools, json
row = next(itertools.islice(load_dataset('HuggingFaceFV/finevideo', split='train', streaming=True), 1))
print(json.dumps(row['json'], indent=2)[:2000])
```

## 3. Build the labeled dataset

Synthetic mutations are deterministic and auto-labeled. The real subset is harvested by captioning each chapter's frame — those samples need **hand-labeling** (`real_is_hallucinated`) before they contribute to metrics.

In [ ]:
from eval.finevideo_loader import make_synthetic_samples, save_dataset

samples = []
for chapter in chapters:
    samples.extend(make_synthetic_samples(chapter))

print(f'{len(samples)} synthetic samples')
save_dataset(samples, Path('/content/eval_dataset.json'))

In [ ]:
# Two verifiers: open-weight Qwen (canonical) + Gemini (closed cross-check).
# Qwen also GENERATES the real-subset captions, so on the real subset
# Qwen-as-verifier is a self-audit and Gemini-as-verifier is a cross-audit.
import os

from eval.captioner import GeminiCaptioner, qwen_captioner
from eval.run_eval import make_frame_for
from vidaudit.vlm.gemini import GeminiBackend
from vidaudit.vlm.qwen_vl import QwenVLBackend

# Gemini needs GEMINI_API_KEY. In Colab, add it to the Secrets sidebar (🔑).
try:
    from google.colab import userdata

    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except Exception:
    pass  # or set os.environ['GEMINI_API_KEY'] = '...' manually

qwen = QwenVLBackend()  # add revision='<sha>' to pin to a commit
gemini = GeminiBackend()

frame_for = make_frame_for(VIDEOS)
generator = qwen_captioner(qwen._runner)  # Qwen produces the real-subset captions

In [ ]:
# Harvest the real subset: Qwen captions each chapter's frame. These come back
# UNLABELED — set `real_is_hallucinated` by hand below.
from eval.finevideo_loader import build_real_samples


def primary_frame(chapter):
    # A FineVideoChapter exposes video_id/timestamp_start/timestamp_end, so it
    # duck-types as the sample make_frame_for expects. Take the primary frame.
    frames = frame_for(chapter)
    return frames[0] if frames else None


real_samples = build_real_samples(chapters, generator, frame_for=primary_frame)
print(f'{len(real_samples)} real samples harvested (label them before scoring).')

### Label the real subset

A caption is a *real hallucination* only if the **frame** doesn't support it — judge against the frame, **not** the ground-truth narration (which spans the whole scene and includes non-visual detail like names). View each frame + caption, then fill in `labels`.

In [ ]:
from IPython.display import display

# Judge each caption against the FRAME (not the narration), then record labels below.
for i, s in enumerate(real_samples):
    frames = frame_for(s)  # EvalSample duck-types like a chapter
    print(f'=== [{i}] {s.video_id} @ {s.timestamp_start:.1f}s ===')
    print('GROUND TRUTH:', s.clean_description)
    print('CAPTION     :', s.mutated_description)
    if frames:
        display(frames[0].resize((480, 270)))
    print()

In [ ]:
# Fill one entry per sample: True if the caption asserts something the FRAME
# doesn't show, False if it's accurate. Unlabeled samples are skipped in scoring.
labels: dict[int, bool] = {
    # 0: False,
    # 1: True,
}
for i, lab in labels.items():
    real_samples[i].real_is_hallucinated = lab

labeled_real = [s for s in real_samples if s.real_is_hallucinated is not None]
print(f'{len(labeled_real)} of {len(real_samples)} real samples labeled.')

## 4. Run the cross-model eval

Each verifier (Qwen, Gemini) gets its own confidence sweep — cheap after the first pass thanks to the verification cache. The text-comparison baseline re-captions with **Gemini**, a *different* model than the Qwen generator, so the real-subset baseline isn't comparing identical text to itself.

In [ ]:
from eval.run_eval import (
    make_baseline_caption_for,
    make_vidaudit_auditor,
    render_cross_model_report,
    run_cross_model_eval,
)

auditors = {
    'qwen': make_vidaudit_auditor(qwen, frame_for),
    'gemini': make_vidaudit_auditor(gemini, frame_for),
}
# Baseline re-captions with Gemini (≠ the Qwen generator) so the real-subset
# baseline doesn't degenerate to comparing identical text.
baseline_caption_for = make_baseline_caption_for(GeminiCaptioner(), frame_for)

all_samples = samples + labeled_real
report = run_cross_model_eval(
    all_samples,
    auditors,
    baseline_caption_for,
    generator_model='qwen',
)

render_cross_model_report(report)
report.save_json(Path('/content/eval_report.json'))
print('\nSaved /content/eval_report.json')

## 5. Per-verifier threshold sweep

Each verifier's shipped default confidence threshold should be its best-F1 point on the synthetic subset, not an asserted constant.

In [ ]:
for v in report.verifiers:
    print(
        f'\n=== {v.verifier}  (best ct={v.best_confidence_threshold:.2f}, '
        f'extraction recall={v.extraction_recall:.2f}) ==='
    )
    print(f'{"conf":>6} {"P":>6} {"R":>6} {"F1":>6}')
    for p in v.sweep:
        c = p.confusion
        print(f'{p.confidence_threshold:6.2f} {c.precision:6.2f} {c.recall:6.2f} {c.f1:6.2f}')